# Vampire Survivors HUD Extractor

This notebook is now organized to save outputs into `results/<video_name>/`. Change `VIDEO_PATH` to run a different video.

Current configured video: `video4_Imelda_100.mp4`.


# Vampire Survivors HUD Extraction - Phase 1
## 1. Project Goal
This notebook is designed to extract Heads-Up Display (HUD) information from Vampire Survivors gameplay videos. By analyzing the video frame-by-frame, we can automatically digitize and log player performance metrics over time.

## 2. Why Phase 1 Focuses on Timer and Enemy Kills
Computer vision tasks can get complex quickly. By starting with just two core metrics:
1. **Run Timer**
2. **Enemy Kill Counter** (next to the skull/skeleton icon)

We establish a robust foundation. These two variables are critical for understanding the progression of a run and serve as perfect anchors for our sub-second sampling strategy.

## 3. Future Expansion Plan
This notebook is designed to be modular and beginner-friendly so it can be easily expanded later. 
Future expansions (Phase 2+) will include:
*   Level indicator
*   Gold readout
*   Pause icon detection
*   Top XP bar tracking
*   Top-left item icon recognition
*   Health bar on the player
*   Floating damage numbers
*   Gem field density and pickup estimation

The code structure cleanly separates **HUD Extractors** (static UI elements) from **Playfield/Event Extractors** (dynamic on-screen objects).

## 4. Imports and Setup
First, we will install EasyOCR and import the necessary libraries. 
*   **OpenCV (`cv2`)**: For reading the video and processing images.
*   **EasyOCR**: A deep-learning based Optical Character Recognition tool to read text.
*   **Pandas**: For structuring our final data and exporting to CSV.
*   **Matplotlib**: For displaying images right here in the notebook.

In [ ]:
!pip install easyocr -q

import cv2
import easyocr
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import re
import time
from pathlib import Path

# Initialize the EasyOCR reader (using English)
# Note: This will download a model the first time it runs.
print("Initializing EasyOCR...")
reader = easyocr.Reader(['en'], gpu=True) # Set gpu=False if CPU-only
print("EasyOCR Ready!")

## 5. Load Video
We will load the video file from your local workspace folder. We use OpenCV's `VideoCapture` to open it.

In [ ]:
VIDEO_PATH = 'video4_Imelda_100.mp4'


def clean_folder_label(text):
    label = re.sub(r'[^A-Za-z0-9._-]+', '_', str(text)).strip('_')
    return label or 'video'


def find_project_dir():
    """Find the kill-counter project folder after MNL is organized."""
    for base in [Path.cwd(), *Path.cwd().parents]:
        if base.name == '01_kill_counter_and_time_stamp':
            return base
        child = base / '01_kill_counter_and_time_stamp'
        if child.exists() and child.is_dir():
            return child
        mnl_child = base / 'MNL' / '01_kill_counter_and_time_stamp'
        if mnl_child.exists() and mnl_child.is_dir():
            return mnl_child
    return Path.cwd()


PROJECT_DIR = find_project_dir()
MNL_DIR = PROJECT_DIR.parent if PROJECT_DIR.name == '01_kill_counter_and_time_stamp' else PROJECT_DIR
VIDEO_DIR = MNL_DIR / '00-Videos'
HUMAN_EXCEL_DIR = MNL_DIR / '00-Human Coded Sheet'


def resolve_video_path(path_text):
    """Find the video from the organized MNL folder or older locations."""
    path = Path(path_text).expanduser()
    candidates = [
        path,
        Path.cwd() / path,
        VIDEO_DIR / path.name,
        MNL_DIR / path.name,
        MNL_DIR / '00-Videos' / path.name,
        PROJECT_DIR / path.name,
        PROJECT_DIR / '00-Videos' / path.name,
        Path.cwd() / 'MNL' / path.name,
        Path.cwd() / 'MNL' / '00-Videos' / path.name,
        Path.cwd().parent / path.name,
        Path.cwd().parent / '00-Videos' / path.name,
        Path.cwd().parent / 'MNL' / path.name,
        Path.cwd().parent / 'MNL' / '00-Videos' / path.name,
    ]
    for candidate in candidates:
        if candidate.exists():
            return candidate
    return path


video_path = resolve_video_path(VIDEO_PATH)
video_name = video_path.stem
video_file = video_path.name

# Every generated file for this video goes here.
OUTPUT_ROOT = PROJECT_DIR / 'results'
VIDEO_OUTPUT_DIR = OUTPUT_ROOT / clean_folder_label(video_name)
DEBUG_OUTPUT_DIR = VIDEO_OUTPUT_DIR / 'debug_plots'
VIDEO_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
DEBUG_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

cap = cv2.VideoCapture(str(video_path))

if not cap.isOpened():
    print(f"Error: Could not open video at {video_path}")
    print("Check VIDEO_PATH above and make sure the video is in the 00-Videos folder.")
else:
    print(f"Successfully opened video: {video_path}")
    print(f"Video name for CSV: {video_name}")
    print(f"Video output folder: {VIDEO_OUTPUT_DIR}")


## 6. Inspect Metadata
Let's check the video's properties (Frames Per Second, Resolution, Total Frames). This helps us calculate exactly which frames correspond to which seconds.

In [ ]:
if cap.isOpened():
    fps = cap.get(cv2.CAP_PROP_FPS)
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    duration_sec = total_frames / fps if fps > 0 else 0
    
    print(f"Video Resolution: {width}x{height}")
    print(f"Frames Per Second (FPS): {fps:.2f}")
    print(f"Total Frames: {total_frames}")
    print(f"Estimated Duration: {duration_sec:.2f} seconds")

## 7. Automatic ROI Calibration (Dynamic)
A **Region of Interest (ROI)** is a bounding box telling the code where to read the HUD.

This section now uses `ROI_MODE = "auto"` by default:
* It scans the current video's top HUD area with OCR to find the timer.
* It uses the skull icon template to place the kill-counter crop just left of the skull.
* It clamps all ROIs to the actual frame size, so different resolutions do not break the crop.

The percentage coordinates below are **fallbacks only**. They are used when automatic detection cannot confidently find the HUD element.

In [ ]:
# ============================================================
# AUTOMATIC ROI CALIBRATION
# ============================================================

ROI_MODE = "auto"  # Options: "auto" or "fallback"

# Fallback ROI coordinates, stored as percentages of the current video size.
# These are used only when auto-detection cannot find the timer or skull icon.
FALLBACK_TIMER_X_PCT   = 0.455
FALLBACK_TIMER_Y_PCT   = 0.045
FALLBACK_TIMER_W_PCT   = 0.090
FALLBACK_TIMER_H_PCT   = 0.050

FALLBACK_KILLS_RIGHT_PCT_16_10 = 0.828
FALLBACK_KILLS_W_PCT_16_10     = 0.065
FALLBACK_KILLS_RIGHT_PCT_16_9  = 0.793
FALLBACK_KILLS_W_PCT_16_9      = 0.060
FALLBACK_KILLS_Y_PCT           = 0.028
FALLBACK_KILLS_H_PCT           = 0.045

SKULL_TEMPLATE_FILES = [
    Path("templates/skull_icon.jpg"),
    Path("templates/skull_anchor.jpg"),
]


def resolve_local_path(path_text):
    """Find helper/template files after the MNL folder is organized."""
    path = Path(path_text).expanduser()
    candidates = [
        path,
        Path.cwd() / path,
        PROJECT_DIR / path,
        PROJECT_DIR / path.name,
        PROJECT_DIR / "templates" / path.name,
        MNL_DIR / path,
        MNL_DIR / path.name,
        MNL_DIR / "01_kill_counter_and_time_stamp" / path,
        MNL_DIR / "01_kill_counter_and_time_stamp" / "templates" / path.name,
        Path.cwd() / "MNL" / path,
        Path.cwd() / "MNL" / "01_kill_counter_and_time_stamp" / path,
        Path.cwd() / "MNL" / "01_kill_counter_and_time_stamp" / "templates" / path.name,
        Path.cwd() / "video games" / path,
        Path.cwd().parent / path,
        Path.cwd().parent / "MNL" / path,
        Path.cwd().parent / "MNL" / "01_kill_counter_and_time_stamp" / path,
        Path.cwd().parent / "MNL" / "01_kill_counter_and_time_stamp" / "templates" / path.name,
        Path.cwd().parent / "video games" / path,
    ]
    for candidate in candidates:
        if candidate.exists():
            return candidate
    return path


def clamp_roi(roi, frame_width, frame_height):
    """Keep an ROI inside the current frame, even when videos have different sizes."""
    x, y, w, h = roi
    x = max(0, min(int(round(x)), frame_width - 1))
    y = max(0, min(int(round(y)), frame_height - 1))
    w = max(1, min(int(round(w)), frame_width - x))
    h = max(1, min(int(round(h)), frame_height - y))
    return x, y, w, h


def get_fallback_rois(frame_width, frame_height):
    """Fallback ROIs based on frame size and aspect ratio."""
    aspect_ratio = frame_width / frame_height if frame_height else 0
    if aspect_ratio >= 1.70:
        kills_right_pct = FALLBACK_KILLS_RIGHT_PCT_16_9
        kills_w_pct = FALLBACK_KILLS_W_PCT_16_9
    else:
        kills_right_pct = FALLBACK_KILLS_RIGHT_PCT_16_10
        kills_w_pct = FALLBACK_KILLS_W_PCT_16_10

    timer_roi = (
        FALLBACK_TIMER_X_PCT * frame_width,
        FALLBACK_TIMER_Y_PCT * frame_height,
        FALLBACK_TIMER_W_PCT * frame_width,
        FALLBACK_TIMER_H_PCT * frame_height,
    )

    kills_w = kills_w_pct * frame_width
    kills_roi = (
        kills_right_pct * frame_width - kills_w,
        FALLBACK_KILLS_Y_PCT * frame_height,
        kills_w,
        FALLBACK_KILLS_H_PCT * frame_height,
    )

    return clamp_roi(timer_roi, frame_width, frame_height), clamp_roi(kills_roi, frame_width, frame_height)


def bbox_to_roi(bbox, frame_width, frame_height, pad_x_pct=0.010, pad_y_pct=0.008):
    x = min(point[0] for point in bbox)
    y = min(point[1] for point in bbox)
    w = max(point[0] for point in bbox) - x
    h = max(point[1] for point in bbox) - y
    pad_x = max(8, frame_width * pad_x_pct)
    pad_y = max(5, frame_height * pad_y_pct)
    return clamp_roi((x - pad_x, y - pad_y, w + 2 * pad_x, h + 2 * pad_y), frame_width, frame_height)


def find_timer_roi(frame, reader):
    """Detect the timer by OCR in the top HUD strip."""
    frame_height, frame_width = frame.shape[:2]
    top_strip_height = min(frame_height, max(120, int(frame_height * 0.18)))
    top_strip = frame[:top_strip_height, :]
    detections = reader.readtext(top_strip, detail=1, allowlist="0123456789:")

    candidates = []
    for bbox, text, conf in detections:
        cleaned = re.sub(r'[^0-9:]', '', text)
        if not re.match(r'^\d{1,2}:\d{2}$', cleaned):
            continue
        roi = bbox_to_roi(bbox, frame_width, frame_height)
        x, y, w, h = roi
        center_x = x + w / 2
        center_penalty = abs(center_x - frame_width / 2) / frame_width
        score = float(conf) - center_penalty
        candidates.append((score, roi, cleaned, float(conf)))

    if not candidates:
        return None, None, 0.0

    score, roi, text, conf = max(candidates, key=lambda item: item[0])
    return roi, text, conf


def load_template(path):
    template_path = resolve_local_path(path)
    template = cv2.imread(str(template_path), cv2.IMREAD_GRAYSCALE)
    return template, template_path


def find_skull_roi(frame):
    """Find the skull icon with multi-scale template matching."""
    frame_height, frame_width = frame.shape[:2]
    top_strip_height = min(frame_height, max(120, int(frame_height * 0.18)))
    search_x0 = int(frame_width * 0.55)
    search_x1 = int(frame_width * 0.95)
    search = frame[:top_strip_height, search_x0:search_x1]
    search_gray = cv2.cvtColor(search, cv2.COLOR_BGR2GRAY)

    best = None
    expected_scale = frame_height / 1800 if frame_height else 1.0
    scales = np.linspace(max(0.35, expected_scale * 0.55), min(1.35, expected_scale * 1.65), 19)

    for template_file in SKULL_TEMPLATE_FILES:
        template, template_path = load_template(template_file)
        if template is None:
            continue

        for scale in scales:
            scaled_w = max(8, int(template.shape[1] * scale))
            scaled_h = max(8, int(template.shape[0] * scale))
            if scaled_w >= search_gray.shape[1] or scaled_h >= search_gray.shape[0]:
                continue

            scaled = cv2.resize(template, (scaled_w, scaled_h), interpolation=cv2.INTER_AREA)
            match = cv2.matchTemplate(search_gray, scaled, cv2.TM_CCOEFF_NORMED)
            _, score, _, loc = cv2.minMaxLoc(match)
            if best is None or score > best[0]:
                best = (score, loc, scaled_w, scaled_h, template_path)

    if best is None:
        return None, 0.0, None

    score, loc, skull_w, skull_h, template_path = best
    if score < 0.35:
        return None, score, template_path

    x = search_x0 + loc[0]
    y = loc[1]
    return clamp_roi((x, y, skull_w, skull_h), frame_width, frame_height), score, template_path


def kill_roi_from_skull(skull_roi, frame_width, frame_height):
    """Place the kill counter ROI immediately left of the skull, excluding the skull itself."""
    skull_x, skull_y, skull_w, skull_h = skull_roi
    gap = max(3, int(frame_width * 0.003))
    left_width = max(int(frame_width * 0.090), skull_w * 5)
    pad_y = max(5, int(frame_height * 0.008))
    roi = (skull_x - gap - left_width, skull_y - pad_y, left_width, skull_h + 2 * pad_y)
    return clamp_roi(roi, frame_width, frame_height)


def auto_calibrate_rois(frame, reader):
    """Auto-detect timer and kill ROIs; fallback to percentage ROIs only when needed."""
    frame_height, frame_width = frame.shape[:2]
    fallback_timer_roi, fallback_kills_roi = get_fallback_rois(frame_width, frame_height)

    if ROI_MODE == "fallback":
        print("ROI_MODE='fallback': using percentage-based fallback ROIs.")
        return fallback_timer_roi, fallback_kills_roi

    timer_roi, timer_text, timer_conf = find_timer_roi(frame, reader)
    skull_roi, skull_score, skull_template_path = find_skull_roi(frame)
    kills_roi = kill_roi_from_skull(skull_roi, frame_width, frame_height) if skull_roi else None

    if timer_roi is None:
        print("Timer auto-detection failed; using fallback timer ROI.")
        timer_roi = fallback_timer_roi
    else:
        print(f"Auto timer ROI: {timer_roi} from '{timer_text}' with confidence {timer_conf:.2f}")

    if kills_roi is None:
        print(f"Skull auto-detection failed; using fallback kill ROI. Best skull score: {skull_score:.2f}")
        kills_roi = fallback_kills_roi
    else:
        print(f"Auto kill ROI: {kills_roi} from skull template {skull_template_path.name} with score {skull_score:.2f}")

    return timer_roi, kills_roi


# ============================================================
# Calibration preview: draw boxes on a real frame
# ============================================================
if cap.isOpened():
    calibration_second = min(10, max(0, duration_sec / 2))
    frame_target = int(calibration_second * fps)
    cap.set(cv2.CAP_PROP_POS_FRAMES, frame_target)
    ret, frame = cap.read()

    if ret:
        height, width = frame.shape[:2]
        TIMER_ROI, KILLS_ROI = auto_calibrate_rois(frame, reader)

        print(f"Resolution: {width}x{height}")
        print(f"Timer ROI (pixels): {TIMER_ROI}")
        print(f"Kills ROI (pixels): {KILLS_ROI}")

        frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        debug_frame = frame_rgb.copy()

        tx, ty, tw, th = TIMER_ROI
        cv2.rectangle(debug_frame, (tx, ty), (tx+tw, ty+th), (255, 0, 0), 4)
        cv2.putText(debug_frame, 'Timer', (tx, max(0, ty-10)),
                    cv2.FONT_HERSHEY_SIMPLEX, 1.2, (255, 0, 0), 3)

        kx, ky, kw, kh = KILLS_ROI
        cv2.rectangle(debug_frame, (kx, ky), (kx+kw, ky+kh), (0, 0, 255), 4)
        cv2.putText(debug_frame, 'Kills', (kx, max(0, ky-10)),
                    cv2.FONT_HERSHEY_SIMPLEX, 1.2, (0, 0, 255), 3)

        plt.figure(figsize=(15, 8))
        plt.imshow(debug_frame)
        plt.title(f"Auto Calibration Frame ({calibration_second:.1f}s mark) - {width}x{height}")
        plt.axis('off')
        plt.show()
    else:
        print("Could not read calibration frame. Re-run the Load Video cell and try again.")

## 8-14. Modular HUD Extraction Pipeline
Here we define the core logic.

### Key Design: HSV Color Masking
Instead of simple grayscale thresholding, we use **HSV color space masking** to isolate 
only the **bright white text pixels** and delete everything else (icons, backgrounds, 
semi-transparent overlays). This prevents the OCR from ever "seeing" the skull icon 
or any other non-text element.

### Key Design: Spike Guard
Even with perfect preprocessing, OCR can occasionally hallucinate. We add a **spike guard**: 
if the detected kill count suddenly jumps by more than 2× the previous value AND the 
previous value was already > 50, we reject the reading as an OCR error.

This section defines helper functions for:
*   Extracting the specific ROIs
*   HSV color masking (isolating white text)
*   Running OCR
*   Cleaning parsed text
*   Temporal validation with spike detection

In [ ]:
# --- HUD EXTRACTOR FUNCTIONS ---

def extract_roi(frame, roi):
    """Extract the image crop based on (x, y, w, h), clamped to the current frame size."""
    frame_height, frame_width = frame.shape[:2]
    x, y, w, h = clamp_roi(roi, frame_width, frame_height)
    return frame[y:y+h, x:x+w]

def preprocess_for_ocr(crop):
    """Preprocess a crop for OCR using proven grayscale + binary threshold.
    
    Pipeline:
    1. Convert BGR → Grayscale
    2. Scale up 2× for better OCR accuracy (larger characters = better recognition)
    3. Apply THRESH_BINARY_INV at threshold 200 to isolate white text
       (Vampire Survivors uses white text with black outlines)
    4. The skull icon is already excluded by the right-anchored ROI,
       so no color masking is needed.
    """
    # Step 1: Convert to grayscale
    gray = cv2.cvtColor(crop, cv2.COLOR_BGR2GRAY)
    
    # Step 2: Scale up 2× for better OCR recognition
    gray = cv2.resize(gray, None, fx=2, fy=2, interpolation=cv2.INTER_CUBIC)
    
    # Step 3: Binary threshold — keeps bright white text, removes everything else
    _, thresh = cv2.threshold(gray, 200, 255, cv2.THRESH_BINARY_INV)
    
    return thresh

def run_ocr(preprocessed_crop, allowlist=None):
    """Perform OCR using EasyOCR."""
    result = reader.readtext(preprocessed_crop, allowlist=allowlist, detail=1)
    if len(result) > 0:
        bbox, text, conf = result[0]
        return text.strip(), conf
    return None, 0.0

def clean_timer_text(text):
    """Clean Timer text. Expecting MM:SS format."""
    if not text:
        return None
    cleaned = re.sub(r'[^0-9:]', '', text)
    if re.match(r'^\d{2}:\d{2}$', cleaned):
        return cleaned
    return None

def clean_kill_text(text):
    """Clean Kill text. Expecting digits only."""
    if not text:
        return None
    cleaned = re.sub(r'[^0-9]', '', text)
    if cleaned.isdigit():
        return int(cleaned)
    return None

def validate_timer(candidates, prev_timer):
    """Temporal validation for Timer. Returns the best MM:SS candidate."""
    valid_candidates = [c for c in candidates if clean_timer_text(c['text']) is not None]
    if valid_candidates:
        best = max(valid_candidates, key=lambda x: x['conf'])
        return clean_timer_text(best['text']), best['conf']
    return None, 0.0

def validate_kills(candidates, prev_kills):
    """Temporal validation for Kills. Kills cannot decrease.
    
    Includes a SPIKE GUARD: limits the maximum kill increase per second.
    In Vampire Survivors, even in late game, kills rarely increase by more
    than ~50-75 per second. We cap it at max(75, 15% of current count).
    
    Examples of what this catches:
      - 296 → 532 (jump of 236, max allowed 75) → REJECTED
      - 532 → 995 (jump of 463, max allowed 80) → REJECTED
      - 460 → 475 (jump of 15, max allowed 75) → ACCEPTED
    """
    valid_candidates = []
    # Maximum allowed increase per second
    max_jump = max(75, int(prev_kills * 0.15))
    
    for c in candidates:
        val = clean_kill_text(c['text'])
        if val is None:
            continue
        # Must be >= previous (non-decreasing rule)
        if val < prev_kills:
            continue
        # SPIKE GUARD: reject if jump exceeds max_jump
        if prev_kills > 10 and (val - prev_kills) > max_jump:
            continue
        valid_candidates.append({'val': val, 'conf': c['conf']})
            
    if valid_candidates:
        # Use MEDIAN instead of MAX — robust to single-sample outliers
        # e.g. median(2725, 2726, 2926) = 2726 (correct!)
        vals = sorted([c['val'] for c in valid_candidates])
        median_val = vals[len(vals) // 2]
        median_conf = [c['conf'] for c in valid_candidates if c['val'] == median_val][0]
        return median_val, median_conf
    return prev_kills, 0.0

## 15. Execute Pipeline and Build Final 1-Second Log
This is the main loop. It samples frames at 0.3s, 0.5s, and 0.7s within each second to generate the final log.

Use the run controls at the top of the next cell:
* Set `RUN_MODE = "minutes"` and `RUN_MINUTES = 2` for a fast 2-minute test.
* Set `RUN_MODE = "seconds"` and change `RUN_SECONDS` for another short test length.
* Set `RUN_MODE = "full"` to process the whole video.

In [ ]:
# Run controls
RUN_MODE = "full"     # Options: "minutes", "seconds", or "full"
RUN_MINUTES = 2       # Used when RUN_MODE = "minutes"
RUN_SECONDS = 120     # Used when RUN_MODE = "seconds"
OUTPUT_CSV = None     # Leave as None for automatic naming, or set a filename/path.

sampling_offsets = [0.3, 0.5, 0.7]  # Fractions of a second to sample
timer_allowlist = '0123456789:'
kills_allowlist = '0123456789'


def format_duration(seconds):
    seconds = max(0, int(round(seconds)))
    minutes, sec = divmod(seconds, 60)
    hours, minutes = divmod(minutes, 60)
    if hours:
        return f"{hours}h {minutes}m {sec}s"
    if minutes:
        return f"{minutes}m {sec}s"
    return f"{sec}s"


def clean_filename_label(text):
    label = Path(text).stem.lower()
    label = re.sub(r'[^a-z0-9]+', '_', label).strip('_')
    return label or 'video'


def get_processing_limit(run_mode, total_seconds):
    if run_mode == "full":
        return total_seconds
    if run_mode == "minutes":
        return min(total_seconds, max(0, int(round(RUN_MINUTES * 60))))
    if run_mode == "seconds":
        return min(total_seconds, max(0, int(RUN_SECONDS)))
    raise ValueError('RUN_MODE must be "minutes", "seconds", or "full".')


def get_run_length_label(processed_seconds, total_seconds):
    if processed_seconds >= total_seconds:
        return "full"
    if processed_seconds % 60 == 0:
        return f"{processed_seconds // 60}min"
    return f"{processed_seconds}s"


def get_output_csv_path(run_mode, processed_seconds, total_seconds):
    if OUTPUT_CSV:
        output_path = Path(OUTPUT_CSV).expanduser()
        if not output_path.is_absolute() and output_path.parent == Path('.'):
            output_path = VIDEO_OUTPUT_DIR / output_path
        return output_path
    video_label = clean_filename_label(video_path)
    length_label = get_run_length_label(processed_seconds, total_seconds)
    return VIDEO_OUTPUT_DIR / f"kill_time_{video_label}_{length_label}.csv"


results = []
prev_timer = None
prev_kills = 0
processed_video_seconds = 0
run_elapsed_seconds = None
estimated_full_run_seconds = None
csv_path = None

if cap.isOpened():
    total_seconds = int(duration_sec)
    limit = get_processing_limit(RUN_MODE, total_seconds)
    processed_video_seconds = limit
    csv_path = get_output_csv_path(RUN_MODE, processed_video_seconds, total_seconds)

    print(f"Video name: {video_name}")
    print(f"Video file: {video_file}")
    print(f"Video length: {format_duration(duration_sec)} ({duration_sec:.2f} seconds)")
    print(f"Processing length: {format_duration(limit)}")
    print(f"Output CSV will be: {csv_path}")
    print(f"Processing {limit} seconds of video...")

    run_started_at = time.perf_counter()

    for sec in range(limit):
        timer_candidates = []
        kills_candidates = []

        # Sample multiple frames around the current second
        for offset in sampling_offsets:
            frame_idx = int((sec + offset) * fps)
            cap.set(cv2.CAP_PROP_POS_FRAMES, frame_idx)
            ret, frame = cap.read()
            if not ret:
                continue

            # Extract ROIs
            t_crop = extract_roi(frame, TIMER_ROI)
            k_crop = extract_roi(frame, KILLS_ROI)

            # Preprocess for OCR
            t_prep = preprocess_for_ocr(t_crop)
            k_prep = preprocess_for_ocr(k_crop)

            # OCR
            t_text, t_conf = run_ocr(t_prep, allowlist=timer_allowlist)
            k_text, k_conf = run_ocr(k_prep, allowlist=kills_allowlist)

            if t_text:
                timer_candidates.append({'text': t_text, 'conf': t_conf})
            if k_text:
                kills_candidates.append({'text': k_text, 'conf': k_conf})

        # Validation
        final_timer, final_t_conf = validate_timer(timer_candidates, prev_timer)
        final_kills, final_k_conf = validate_kills(kills_candidates, prev_kills)

        results.append({
            'video_name': video_name,
            'video_file': video_file,
            'video_duration_minutes': round(duration_sec / 60, 3),
            'processed_minutes': round(processed_video_seconds / 60, 3),
            'Video Second': sec,
            'Time Stamp': final_timer,
            'Kill Counter Quantity': final_kills,
            'review_flag': final_timer is None
        })

        # Update state
        if final_timer:
            prev_timer = final_timer
        if final_kills is not None:
            prev_kills = final_kills

        if sec % 10 == 0 and sec > 0:
            print(f"Processed up to {sec} seconds...")

    run_elapsed_seconds = time.perf_counter() - run_started_at
    if processed_video_seconds and processed_video_seconds < total_seconds:
        estimated_full_run_seconds = (run_elapsed_seconds / processed_video_seconds) * total_seconds

    print("Processing Complete!")
else:
    print("Error: Video is not open. Re-run the Load Video cell first.")


## 16. Export CSV and Runtime Summary
This cell displays the extracted rows, saves the CSV, and prints how long the run took.

The CSV filename is automatic unless you set `OUTPUT_CSV` in the previous cell.

In [ ]:
df = pd.DataFrame(results)

# Display first 20 rows
display(df.head(20))

# Save to this video's organized output folder.
csv_path = Path(csv_path)
csv_path.parent.mkdir(parents=True, exist_ok=True)
df.to_csv(csv_path, index=False)
print()
print(f"Saved {len(df)} rows to {csv_path}")

if run_elapsed_seconds is not None:
    print()
    print("Run summary")
    print(f"- Video name: {video_name}")
    print(f"- Video file: {video_file}")
    print(f"- Video length: {format_duration(duration_sec)}")
    print(f"- Processed video: {format_duration(processed_video_seconds)}")
    print(f"- Run time: {format_duration(run_elapsed_seconds)} ({run_elapsed_seconds:.1f} seconds)")
    if estimated_full_run_seconds is not None:
        print(f"- Estimated full-video run time: {format_duration(estimated_full_run_seconds)}")


## 17. Debug Previews
Run this cell to see exactly what the HSV color masking produces — the preprocessed 
images that EasyOCR actually reads. You should see clean white text on a black background 
with no skull icon visible.

In [ ]:
# Show preprocessing results from a sample frame (at the 30-second mark)
if cap.isOpened():
    cap.set(cv2.CAP_PROP_POS_FRAMES, int(30 * fps))
    ret, sample_frame = cap.read()
    
    if ret:
        t_crop = extract_roi(sample_frame, TIMER_ROI)
        k_crop = extract_roi(sample_frame, KILLS_ROI)
        t_prep = preprocess_for_ocr(t_crop)
        k_prep = preprocess_for_ocr(k_crop)
        
        fig, axs = plt.subplots(2, 2, figsize=(14, 6))
        
        # Raw crops
        axs[0][0].imshow(cv2.cvtColor(t_crop, cv2.COLOR_BGR2RGB))
        axs[0][0].set_title("Timer: Raw Crop")
        axs[0][0].axis('off')
        
        axs[0][1].imshow(cv2.cvtColor(k_crop, cv2.COLOR_BGR2RGB))
        axs[0][1].set_title("Kills: Raw Crop")
        axs[0][1].axis('off')
        
        # After HSV color masking
        axs[1][0].imshow(t_prep, cmap='gray')
        axs[1][0].set_title("Timer: After HSV Masking (what OCR sees)")
        axs[1][0].axis('off')
        
        axs[1][1].imshow(k_prep, cmap='gray')
        axs[1][1].set_title("Kills: After HSV Masking (what OCR sees)")
        axs[1][1].axis('off')
        
        plt.tight_layout()
        preview_path = DEBUG_OUTPUT_DIR / 'preprocessing_preview.jpg'
        preview_path.parent.mkdir(parents=True, exist_ok=True)
        plt.savefig(preview_path, dpi=150)
        plt.show()
        print(f"Preview saved to {preview_path}")

# Release the video capture object when done
cap.release()


## 18. Notes for Future HUD & Playfield Modules
To maintain this modular architecture in Phase 2, follow these steps:
1. **New ROIs**: Define new normalized percentage coordinates at the top of Section 7 (e.g., `GOLD_X_PCT`, `LEVEL_Y_PCT`). The `get_pixel_rois()` function handles the conversion automatically.
2. **New Extractors**: Add specific extraction/cleaning functions in Section 8-14 (e.g., `clean_gold_text()`).
3. **Playfield Events**: For non-static elements (like floating damage numbers), object detection models (like YOLO) or contour-based tracking will be needed instead of static ROIs.

### Technical Notes
*   **Resolution Independence**: All ROIs use normalized coordinates (0.0 to 1.0). The notebook auto-converts to pixels based on the video's actual resolution.
*   **Right-Anchored Kills**: The kill counter ROI is anchored from its right edge (just before the skull icon) and extends left. This ensures the skull is never included regardless of how many digits the kill count has.
*   **HSV Color Masking**: The preprocessing pipeline isolates only bright white pixels, automatically removing dark icons (skull, items) and background elements before OCR.

## Create 5-Second Excel Columns

This section converts the raw per-second OCR CSV into coding-sheet-friendly 5-second intervals.

Logic:
* Keep the original per-second CSV as raw data.
* Add two endpoint columns where rows between endpoints are blank and every 5-second endpoint gets a value.
* Also create a compact interval table with one row per interval.

Example: interval `0:05-0:10` uses the OCR value at `Video Second == 10`.

In [ ]:
# Create 5-second endpoint columns and a compact interval table for Excel/Google Sheets
SOURCE_CSV = None  # None uses csv_path from the current run. Or set a CSV path manually.
INTERVAL_SECONDS = 5


def resolve_table_path(path):
    path = Path(path).expanduser()
    candidates = [
        path,
        Path.cwd() / path,
        PROJECT_DIR / path,
        PROJECT_DIR / path.name,
        VIDEO_OUTPUT_DIR / path.name,
        PROJECT_DIR / 'results' / video_name / path.name,
        MNL_DIR / path,
        MNL_DIR / path.name,
        MNL_DIR / 'results' / video_name / path.name,
        Path.cwd() / 'MNL' / path.name,
        Path.cwd() / 'MNL' / 'results' / video_name / path.name,
        Path.cwd() / 'MNL' / '01_kill_counter_and_time_stamp' / 'results' / video_name / path.name,
        Path.cwd().parent / path.name,
        Path.cwd().parent / 'results' / video_name / path.name,
        Path.cwd().parent / 'MNL' / 'results' / video_name / path.name,
        Path.cwd().parent / 'MNL' / '01_kill_counter_and_time_stamp' / 'results' / video_name / path.name,
    ]
    for candidate in candidates:
        if candidate.exists():
            return candidate
    return path


def get_source_csv_path():
    if SOURCE_CSV is not None:
        return resolve_table_path(SOURCE_CSV)
    if "csv_path" in globals() and csv_path:
        return Path(csv_path)

    video_label = clean_filename_label(video_path)
    guesses = [
        VIDEO_OUTPUT_DIR / f"kill_time_{video_label}_full.csv",
        PROJECT_DIR / 'results' / video_name / f"kill_time_{video_label}_full.csv",
        MNL_DIR / 'results' / video_name / f"kill_time_{video_label}_full.csv",
        MNL_DIR / f"kill_time_{video_label}_full.csv",
    ]
    for guess in guesses:
        resolved = resolve_table_path(guess)
        if resolved.exists():
            return resolved
    return guesses[0]


def mmss(total_seconds):
    total_seconds = int(total_seconds)
    minutes, seconds = divmod(total_seconds, 60)
    return f"{minutes}:{seconds:02d}"


def interval_label(start_second, end_second):
    return f"{mmss(start_second)}-{mmss(end_second)}"


source_csv = resolve_table_path(get_source_csv_path())
raw = pd.read_csv(source_csv)
raw["Video Second"] = pd.to_numeric(raw["Video Second"], errors="coerce").astype("Int64")
raw["Kill Counter Quantity"] = pd.to_numeric(raw["Kill Counter Quantity"], errors="coerce")

# These two columns are made for the coding sheet. Non-endpoint rows stay blank.
raw["Excel Time Stamp"] = pd.NA
raw["Excel Kill Counter Quantity"] = pd.NA

valid_seconds = raw["Video Second"].dropna().astype(int)
max_video_second = int(valid_seconds.max())
last_endpoint = int(np.ceil((max_video_second + 1) / INTERVAL_SECONDS) * INTERVAL_SECONDS)

interval_rows = []
for end_second in range(INTERVAL_SECONDS, last_endpoint + 1, INTERVAL_SECONDS):
    start_second = end_second - INTERVAL_SECONDS
    available = raw[raw["Video Second"] <= end_second]
    if available.empty:
        continue

    # Exact endpoint is preferred. If the endpoint is beyond the CSV, use nearest previous second.
    exact = raw[raw["Video Second"] == end_second]
    if not exact.empty:
        source_row_index = exact.index[-1]
        matched_second = end_second
    else:
        source_row_index = available.index[-1]
        matched_second = int(raw.loc[source_row_index, "Video Second"])

    label = interval_label(start_second, end_second)
    kill_value = raw.loc[source_row_index, "Kill Counter Quantity"]
    source_time_stamp = raw.loc[source_row_index, "Time Stamp"] if "Time Stamp" in raw.columns else pd.NA

    raw.loc[source_row_index, "Excel Time Stamp"] = label
    raw.loc[source_row_index, "Excel Kill Counter Quantity"] = kill_value

    interval_rows.append({
        "Time Stamp": label,
        "Interval Start Second": start_second,
        "Interval End Second": end_second,
        "Matched Video Second": matched_second,
        "Source OCR Time Stamp": source_time_stamp,
        "Kill Counter Quantity": kill_value,
    })

interval_df = pd.DataFrame(interval_rows)

endpoint_output = VIDEO_OUTPUT_DIR / f"{source_csv.stem}_with_excel_endpoint_columns.csv"
interval_output = VIDEO_OUTPUT_DIR / f"{source_csv.stem}_5sec_intervals.csv"
endpoint_output.parent.mkdir(parents=True, exist_ok=True)
interval_output.parent.mkdir(parents=True, exist_ok=True)

raw.to_csv(endpoint_output, index=False)
interval_df.to_csv(interval_output, index=False)

print(f"Source CSV: {source_csv}")
print(f"Saved per-second file with endpoint columns to: {endpoint_output}")
print(f"Saved compact 5-second interval file to: {interval_output}")
print(f"Created {len(interval_df)} interval rows.")

display(raw[["Video Second", "Time Stamp", "Kill Counter Quantity", "Excel Time Stamp", "Excel Kill Counter Quantity"]].head(15))
display(interval_df.head(12))
